In [ ]:
!pip install pytorch_lightning

In [2]:
import os
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping
from torchmetrics import MetricCollection, F1Score, AUROC

Использую стандартную нормализацию со средним 0.5 и отклонением 0.5, так как FashionMNIST состоит из черно-белых изображений, и это приводит значения пикселей к диапазону [-1, 1], что ускоряет сходимость. Данные разбиты на train (54000) и validation (6000), чтобы оставить достаточно данных для обучения, но иметь репрезентативную валидацию (10%).

In [3]:
class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = "./data", batch_size: int = 64):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def prepare_data(self):
        datasets.FashionMNIST(self.data_dir, train=True, download=True)
        datasets.FashionMNIST(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            full_train = datasets.FashionMNIST(
                self.data_dir, train=True, transform=self.transform
            )
            self.train_set, self.val_set = random_split(
                full_train, [54000, 6000], generator=torch.Generator().manual_seed(42)
            )

        if stage == "test" or stage is None:
            self.test_set = datasets.FashionMNIST(
                self.data_dir, train=False, transform=self.transform
            )

    def train_dataloader(self):
        return DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True, num_workers=2)

    def val_dataloader(self):
        return DataLoader(self.val_set, batch_size=self.batch_size, num_workers=2)

    def test_dataloader(self):
        return DataLoader(self.test_set, batch_size=self.batch_size, num_workers=2)


In [4]:
pl.seed_everything(42)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

Обоснование архитектуры и гиперпараметров:

Архитектура: Выбрана сверточная нейросеть (CNN), так как она учитывает пространственную структуру изображений, что критично для задач компьютерного зрения. Использованы два блока сверток (Conv2d + ReLU + MaxPool) для извлечения иерархических признаков, за которыми следует полносвязный классификатор. Количество каналов (16, 32) выбрано небольшим для предотвращения переобучения на простых картинках 28x28.

Оптимизатор: Использован AdamW с lr=1e-3. AdamW обычно показывает более быструю сходимость и лучшую генерализацию, чем обычный SGD, и требует меньше настройки гиперпараметров.



In [5]:
class FashionMNISTModel(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        metrics = MetricCollection([
            F1Score(task="multiclass", num_classes=10, average="weighted"),
            AUROC(task="multiclass", num_classes=10)
        ])
        self.val_metrics = metrics.clone(prefix="val_")
        self.test_metrics = metrics.clone(prefix="test_")

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        self.val_metrics.update(logits, y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        output = self.val_metrics.compute()
        self.log_dict(output)
        self.val_metrics.reset()

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.test_metrics.update(logits, y)
        self.log("test_loss", loss)
        return loss

    def on_test_epoch_end(self):
        output = self.test_metrics.compute()
        self.log_dict(output)
        self.test_metrics.reset()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)


In [6]:
data_module = FashionMNISTDataModule()
model = FashionMNISTModel(lr=0.001)

In [7]:
logger = TensorBoardLogger("tb_logs", name="fashion_mnist_cnn")

In [8]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    verbose=True,
    mode="min"
)

In [9]:
trainer = pl.Trainer(
    max_epochs=15,
    accelerator="auto", # GPU если есть
    devices=1,
    logger=logger,
    callbacks=[early_stop_callback],
    log_every_n_steps=50
)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


In [10]:
trainer.fit(model, data_module)

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 326kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.65MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 26.6MB/s]
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features     │ Sequential       │  4.8 K │ train │     0 │
│ 1 │ classifier   │ Sequential       │  202 K │ train │     0 │
│ 2 │ val_metrics  │ MetricCollection │      0 │ train │     0 │
│ 3 │ test_metrics │ MetricCollection │      0 │ train │     0 │
└───┴──────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 206 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 206 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 18                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved. New best score: 0.383
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.083 >= min_delta = 0.0. New best score: 0.300
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.025 >= min_delta = 0.0. New best score: 0.275
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.011 >= min_delta = 0.0. New best score: 0.264
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.017 >= min_delta = 0.0. New best score: 0.247
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.011 >= min_delta = 0.0. New best score: 0.236
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.003 >= min_delta = 0.0. New best score: 0.233
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.004 >= min_delta = 0.0. New best score: 0.228
INFO:pytorch_lightning.callbacks.ear

In [11]:
trainer.test(model, data_module)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   test_MulticlassAUROC    │    0.9946050643920898     │
│  test_MulticlassF1Score   │    0.9127564430236816     │
│         test_loss         │    0.2750991880893707     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.2750991880893707,
  'test_MulticlassF1Score': 0.9127564430236816,
  'test_MulticlassAUROC': 0.9946050643920898}]

Итог:

Сходимость: Модель быстро сходилась, val_loss уменьшался до 11-й эпохи.

Early Stopping: Механизм ранней остановки сработал на 14-й эпохе, так как val_loss перестал улучшаться в течение 3 эпох (лучший результат был зафиксирован около 0.228). Это предотвратило переобучение модели.

Качество на тесте:

F1-score: ~0.913 — отличный результат для базовой модели, показывающий высокий баланс точности и полноты по всем классам.

ROC AUC: ~0.995 — модель почти идеально разделяет классы (вероятность правильного ранжирования случайной положительной пары выше отрицательной близка к 1).

Loss: 0.275 — значение на тесте близко к валидационному, что говорит о хорошей обобщающей способности.